<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

# Your Places, Week 7 Session 1 - Buffers and Spatial Joins

**CP101 · GeoPandas II**

This activity is worth **2.5 bonus points for class participation**.
Part A, in class, is worth 0.5 points, and Part B, at home, is worth 2.

In Lab 2, each of you submitted a My Places file.
We combined your files into one table, `class_entries.csv`, dropped the entries without a `love it`, `like it` or `just okay` rating, and fixed most of the other problems we found along the way.
A few discrepancies are left for you, such as one place spelled in several ways.

Last week you put places on a map as points.
Today you will clean what is left, build one row per place, and use buffers and spatial joins to find the class's hotspots for eating, studying and unwinding.

| Tool | Use |
|---|---|
| `.buffer(distance)` | Draw an area within a distance of a geometry |
| `.within(shape)` | Test whether each geometry is inside a shape |
| `.dissolve(by=column)` | Combine geometries that share a value |
| `gpd.sjoin(...)` | Join layers using a spatial relationship |

In **Part A**, in class, you follow the walkthrough, try two distance experiments, and map the clusters and places together.
In **Part B**, an optional bonus at home, you repeat the analysis on your own places.

**What you submit**

- **In class.** Submit Part A to the [Gradescope assignment](https://www.gradescope.com/courses/1333040/assignments/8805378) before **2:10 pm**, for **0.5 bonus points**.
  The cells at the end of Part A pack this notebook and the Part A maps saved in `vis/` into a zip, and they stop with a message if a Your Turn is not done.
- **Optional, for 2 bonus points.** Complete Part B at home and submit it to the [bonus assignment on Gradescope](https://www.gradescope.com/courses/1333040/assignments/8805589) by **Wednesday, October 7, at 12:30 pm**.
  The cells at the end of Part B make its zip, and they stop with a message if a question is not done.

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Setup

Run the installation cell once if these packages are missing, then import the libraries.
`%pip` installs into this notebook's Python environment.

In [ ]:
%pip install "geopandas[all]" -q

In [ ]:
from pathlib import Path

import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

output_dir_vis = Path("./vis")
output_dir_vis.mkdir(exist_ok=True)

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

# Part A: Walkthrough

## 1. Load your class's places

Each row of `data/raw/class_entries.csv` is an **entry**, a place listed by one student.
Several students can list the same place.

The prepared file uses anonymous student numbers.
It leaves out residences, locations we could not verify, and places outside Berkeley and the UC campus.

It also leaves out the **165 entries** without a `love it`, `like it` or `just okay` rating.
Of these, 77 had a blank rating and 88 used a number scale such as `8` or `7/10`, which cannot be compared with the three labels.
As a result, 28 students have no entries in this file, and their places do not appear in the maps below.

| Column | Meaning |
|---|---|
| `student` | anonymous student number |
| `name`, `category` | place name and category, such as `food` or `reading` |
| `address` | address as entered by the student |
| `lat`, `lon` | checked coordinates in degrees; see section 2 |
| `rating` | `love it`, `like it` or `just okay` |

In [ ]:
df = pd.read_csv("data/raw/class_entries.csv")

print("rows:", len(df))
print("students:", df["student"].nunique())
print("different place names:", df["name"].nunique())
df.head()

The file has **351 entries from 75 students**, with **168 name spellings**.
Keep `df` in pandas while we clean and summarise it; create geometries only after we have one row per place.

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 2. Clean and prepare the data

Doe Library appears in **18 rows** of `df`, and in 5 more spelled `Doe Memorial Library`.
All 23 rows are the same building.
If each row became its own point on the map, Doe Library would be counted 23 times whenever we count the places near something.

The fix is to keep **two tables** that share the place name:

- **`df`, the entries table**, has one row per entry, meaning one student listing one place.
  It keeps each student's own category and rating.
- **`places`, the places table**, has one row per place.
  It holds the place's location and summaries of its entries, such as how many students listed it.

This is how **relational databases** organize data.
Their goal is to minimize **redundancy**, which means storing the same information in many rows.
Each fact is stored once, in the table it describes.
What a student said goes in the entries table, and what belongs to the place itself, such as its location, goes in the places table.
Without this split, Doe Library's point would be repeated in 23 rows.

Here are three of the 23 Doe Library entries: 

| `student` | `name` | `rating` |
|---|---|---|
| 15 | Doe Library | like it |
| 16 | Doe Library | love it |
| 20 | Doe Memorial Library | like it |

<br>

All those variations should merge into one entry:

| `name` | `popularity` | `share_loved` | `geometry` |
|---|---|---|---|
| Doe Library | 23 | 0.48 | one point |

Nothing is lost by collapsing the rows, because `popularity` keeps the number of students who listed the place.
Each place now has **one point**, so buffers and spatial joins measure it once, and a wrong location has to be fixed in only one row.

The cleaned `name` links the two tables.
Names work here, but two branches of one shop can share a name.
This file keeps them apart by adding the street, as in `My-O-My (Euclid Ave)` and `My-O-My (Telegraph Ave)`; larger datasets give each place a unique ID instead.

We work in pandas until the places table is finished, and only then add geometry:

1. Clean the names, so each place has one spelling.
2. Check that each unique place has one location.
3. Turn the categorical ratings into numbers.
4. Summarise the entries into one row per place.
5. Turn that summary into a GeoDataFrame called `places`.

**Step 1: clean the names.**
`groupby("name")` treats `RSF` and `Recreational Sports Facility` as different places.
Look for these problems:

| Problem | Example |
|---|---|
| Spelling, capitals or abbreviations | `Moffit Library`, `GARA` / `Gara`, `RSF` |
| One place, different names or addresses | `Doe Library` / `Doe Memorial Library`; Campanile Way / University Dr |
| Different places, one street address | `Boba Ninja` and `Gypsy's Trattoria Italiana` in the same food court |
| Similar names, different places | `Berkeley Bowl` and `Berkeley Bowl West` |
| Coordinates disagree with the address | one Games of Berkeley point was about 0.7 miles away |
| Other inconsistent fields | vague addresses, different categories or rating scales |

Most of these were fixed while creating `class_entries.csv`, including most spelling variants.
A few variants were left in for practice.
Step 2 checks locations; here we standardize names, check shared addresses, and read the sorted names.

**First, standardize the names.**
`.str.strip()` removes surrounding spaces in `df`.
A lowercase copy from `.str.lower()` helps find capitalization variants without changing the displayed names.
The file's names are already trimmed, but keep this step when cleaning new files.

In [ ]:
df["name"] = df["name"].str.strip()        # remove spaces at the start and end
name_lower = df["name"].str.lower()

print("different names:", df["name"].nunique())
print("different names in lowercase:", name_lower.nunique())

# the spellings that share one lowercase name
spellings = df.groupby(name_lower)["name"].unique()
spellings[spellings.str.len() > 1]

`GARA` and `Gara` differ only in capitalization.
Most whitespace and capitalization problems were fixed during file preparation.

**Second, check names that share an address.**
Different businesses or units can share a street address.
An address match is a candidate to review, not proof that two entries are the same place.

Normalize a copy of the address by lowercasing it, replacing commas with spaces, removing periods, and collapsing repeated spaces.
Preserve suite and unit identifiers when supplied.
Then list addresses with more than one name.

In [ ]:
df["address_clean"] = (
    df["address"]
    .str.lower()
    .str.replace(",", " ")      # a space, because some addresses have no space after a comma ("Ave,Berkeley")
    .str.replace(".", "")       # remove periods, so "Ave." and "Ave" match
    .str.split()
    .str.join(" ")              # split on spaces and rejoin, so repeated spaces become one
)

names_per_address = df.groupby("address_clean").agg(
    names=("name", "unique"),
    different_names=("name", "nunique"),
)
names_per_address[names_per_address["different_names"] > 1]

The check finds **15 addresses with multiple names**.
Seven contain confirmed spelling variants; eight refer to distinct or ambiguous places.
Examples include two food-court businesses, a library inside Haas, and different places entered simply as `Berkeley, CA 94720`.

Compare names and full addresses, including suite or unit details.
Check missing details before combining names.

**Third, read the sorted names.**
This can reveal variants that use different addresses.
Similar spellings often appear near each other; abbreviations may appear far apart.

In [ ]:
sorted(df["name"].unique())

The list reveals `Doe Memorial Library`, which the address check missed.
It also contains distinct stores such as `Berkeley Bowl` and `Berkeley Bowl West`.

**Apply only confirmed matches.**
The dictionary maps reviewed variants to one spelling; `.replace()` updates `df` before grouping.
Larger datasets may use text-similarity tools to suggest matches, but still need review.

In [ ]:
name_variation_fix = {
    "Cheeseboard": "Cheese Board Pizza",
    "Cheeseboard Collective": "Cheese Board Pizza",
    "Cheese Board Collective Pizzeria": "Cheese Board Pizza",
    "California Football Stadium": "California Memorial Stadium",
    "Calirfornia Memorial Stadium": "California Memorial Stadium",
    "RSF": "Recreational Sports Facility",
    "Mezzos": "Mezzo",
    "Moffit Library": "Moffitt Library",
    "GARA": "Gara",
    "Memorial Galde": "Memorial Glade",
    "Doe Memorial Library": "Doe Library",
}

df["name"] = df["name"].replace(name_variation_fix)

listed_by = df.groupby("name")["student"].nunique()
print("different place names:", df["name"].nunique())
print("places listed by more than one student:", (listed_by > 1).sum())

After cleaning, there are **157 places**, and **60** were listed by multiple students.
Combining the Doe Library spellings brings its popularity to **23 students**.

**Step 2: check one location per place.**
**Geocoding** converts an address to a location.
While preparing this file, we geocoded addresses with Nominatim, checked the results against the submissions, and excluded locations we could not verify.

Geocoders can return a street or city instead of a building.
Review the result before using it.
The example below stays commented out because the public [Nominatim service](https://operations.osmfoundation.org/policies/nominatim/) limits requests; in class, use the supplied, checked coordinates.

In [ ]:
# Geocode one address per place. Commented out so the class does not overload the free server.

# place_addresses = df.groupby("name")["address"].first()
#
# geocoded = gpd.tools.geocode(
#     place_addresses,
#     provider="nominatim",
#     user_agent="cp101-your-places",
# )

Check for missing or invalid coordinates, then confirm that every cleaned name has exactly one latitude and one longitude.
`assert` stops the notebook with an error if its condition is `False`.

`pd.to_numeric()` turns each value into a number.
By default it stops with a `ValueError` at the first value it cannot read, such as `"37.87°N"`.
With `errors="coerce"`, that value becomes `NaN` (missing) instead, and `.between()` returns `False` for `NaN`, so the `assert` lines catch it.

The check for one location per place takes three steps:

1. `df.groupby("name")[["lat", "lon"]].nunique(dropna=False)` counts, for each place, how many different latitudes and longitudes its entries have.
   `dropna=False` counts a missing value as one more value.
2. `.eq(1)` marks each count that is exactly 1 with `True`.
3. The first `.all()` checks each column, giving one `True` or `False` for `lat` and one for `lon`.
   The second `.all()` combines those two into a single answer for the `assert`.

`coordinate_counts.max()` then shows the largest count in each column.
If both are 1, every place has exactly one location.

When all entries for a place share the same coordinates, that still does not prove the location is correct.

In [ ]:
# errors="coerce" turns anything that is not a number into NaN, which the asserts below catch
df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
assert df["lat"].between(-90, 90).all(), "Fix missing or invalid latitudes."
assert df["lon"].between(-180, 180).all(), "Fix missing or invalid longitudes."

coordinate_counts = df.groupby("name")[["lat", "lon"]].nunique(dropna=False)
assert coordinate_counts.eq(1).all().all(), "Resolve conflicting locations before grouping."
coordinate_counts.max()

Both counts are 1.
We can keep one coordinate pair per place when summarising.

**Step 3: encode ratings.**
`rating` is **ordinal** data, because its three labels have an order, `just okay` < `like it` < `love it` (review the [Week 5 slide on nominal and ordinal data](https://cdn.jsdelivr.net/gh/UCB-CP101/materials-fa26@main/slides/w05/CP101_f26_w05_s1_sep21.pdf#page=26)). pandas has a type for this.
`pd.Categorical(..., categories=..., ordered=True)` stores the labels with their order, and any label that is not in the list becomes a missing value, which the `assert` catches.

`.cat.codes` numbers the categories in that order, starting at 0, so adding 1 gives `1 = just okay`, `2 = like it`, `3 = love it`.
`pd.get_dummies()` makes one **dummy variable** per category, a column that is 1 when the entry has that rating and 0 otherwise.
We keep the `love it` column as `is_loved`; its mean is the share of entries that said `love it`.

A mean rating assumes equal gaps between these labels.
Use the median and share loved alongside it.

In [ ]:
rating_order = ["just okay", "like it", "love it"]

df["rating"] = pd.Categorical(
    df["rating"],
    categories=rating_order,
    ordered=True,
)
assert df["rating"].notna().all(), "Review unknown or missing ratings."

df["rating_num"] = df["rating"].cat.codes + 1     # codes start at 0, so add 1

rating_dummies = pd.get_dummies(
    df["rating"],
    dtype=int,
)
df["is_loved"] = rating_dummies["love it"]

df[["name", "rating", "rating_num", "is_loved"]].head()

**Step 4: summarise by place.**
`.groupby("name").agg(...)` computes one value per place for each new column.

| Column | Calculation |
|---|---|
| `lat`, `lon` | first coordinate pair, after checking agreement |
| `popularity` | number of distinct students listing that place |
| `rating_mean`, `rating_median` | average and middle score |
| `rating_min`, `rating_max` | lowest and highest score |
| `share_loved` | mean of the 0/1 `is_loved` column |
| `eat`, `study`, `unwind` | number of entries that listed the place as each place type |

**How do students use each place?**
`category` is **nominal** data.
Its values, such as `food` and `reading`, have no order, so unlike `rating` they cannot be turned into scores or averaged (review the [Week 5 slide on nominal and ordinal data](https://cdn.jsdelivr.net/gh/UCB-CP101/materials-fa26@main/slides/w05/CP101_f26_w05_s1_sep21.pdf#page=26)).
Students also do not always agree.
Memorial Glade was listed 9 times for unwinding and twice for studying.
A single label would hide one of those uses, so we count the entries of each kind instead.

First, group the seven categories into three **place types**:

| Place type | Categories |
|---|---|
| `eat` | `food`, `dessert` |
| `study` | `reading`, `bookstore` |
| `unwind` | `relaxing`, `sport`, `gaming` |

Then count the entries of each type for each place:

1. `.groupby(["name", "place_type"]).size()` counts the entries for each place and type, with one row per pair.
2. `.unstack(fill_value=0)` moves the types into columns, so there is one row per place, with 0 where a place has no entries of a type.
3. `.join()` adds the three columns to the summary, matching rows on `name`.

For two places, `.size()` gives

| `name` | `place_type` | count |
|---|---|---|
| Grizzly Peak Vista Point | study | 1 |
| Grizzly Peak Vista Point | unwind | 1 |
| Memorial Glade | study | 2 |
| Memorial Glade | unwind | 9 |

and `.unstack(fill_value=0)` turns it into

| `name` | `eat` | `study` | `unwind` |
|---|---|---|---|
| Grizzly Peak Vista Point | 0 | 1 | 1 |
| Memorial Glade | 0 | 2 | 9 |

From here on, a place counts as a study place if at least one entry listed it for studying, and the same for the other types.
Seven places have entries of two types, so they count as both.

In [ ]:
place_summary = df.groupby("name").agg(
    lat=("lat", "first"),
    lon=("lon", "first"),
    popularity=("student", "nunique"),
    rating_mean=("rating_num", "mean"),
    rating_median=("rating_num", "median"),
    rating_min=("rating_num", "min"),
    rating_max=("rating_num", "max"),
    share_loved=("is_loved", "mean"),
)

place_types = {
    "food": "eat",
    "dessert": "eat",
    "reading": "study",
    "bookstore": "study",
    "relaxing": "unwind",
    "sport": "unwind",
    "gaming": "unwind",
}
types = ["eat", "study", "unwind"]
df["place_type"] = df["category"].map(place_types)

# entries per place and type, then one column per type
type_counts = (
    df.groupby(["name", "place_type"])
    .size()
    .unstack(fill_value=0)
)
place_summary = place_summary.join(type_counts).reset_index()

place_summary = place_summary.sort_values(
    ["popularity", "name"], ascending=[False, True],
).reset_index(drop=True)
print(len(place_summary), "places")
place_summary.head(8)

The summary has **157 rows**, one per place.
Doe Library leads with 23 students.

The mean and the median can give different pictures of the same place.
Yogurt Park has 13 entries, with 7 `love it`, 2 `like it` and 4 `just okay`.
Its median is 3, because more than half of its entries say `love it`.
Its mean is 2.2, because the four `just okay` entries pull the average down.
Opinions about Yogurt Park are split, and neither number shows that on its own, so read `share_loved` (0.54) and `rating_min` (1) alongside them.

<hr style="border: 1px solid #fdb515;" />

### ✏️ Your Turn: one rating for two uses

`rating_mean` and `share_loved` average all of a place's entries together.
Memorial Glade has 9 entries for unwinding and 2 for studying, so its `rating_mean` of 2.73 mixes how students rate it as a place to relax with how they rate it as a place to read.

💡 **Should those ratings be averaged into one number?**
In one or two sentences, say what a single average could hide and how you would fix it.
Then fill in the `groupby` below with two columns, so each use of a place gets its own ratings.

In [ ]:
type_ratings = df.groupby([..., ...]).agg(
    entries=("student", "size"),
    rating_mean=("rating_num", "mean"),
    share_loved=("is_loved", "mean"),
)
type_ratings.loc[["Memorial Glade", "Greek Theatre"]]

*Your answer:*

<details>
<summary>Check your answer</summary>

`df.groupby(["name", "place_type"])` gives one row for each use of a place.
A single average hides that a place can be good for one use and not for another.
The Greek Theatre's overall mean of 2.25 combines a 3.0 from the one student who studies there and a 2.0 from the three who unwind there.
Memorial Glade's two study entries average 3.0, and its nine unwind entries average 2.67.
Keep `entries` next to each rating, because a rating from one or two entries is weak evidence.

</details>

**Step 5: create the GeoDataFrame.**
Build one point for each row of `place_summary`, with **x = lon and y = lat**.
`crs=4326` identifies the input coordinates as longitude and latitude in degrees.

`places` becomes the spatial reference table; `df` remains the individual responses in pandas.

In [ ]:
place_points = gpd.points_from_xy(
    x=place_summary["lon"],
    y=place_summary["lat"],
)

places = gpd.GeoDataFrame(
    place_summary,
    geometry=place_points,
    crs=4326,
)

print(len(places), "places, one point each")
places[["name", "popularity", "rating_mean", "geometry"]].head()

Map the cleaned places, colored by popularity, with pale red for a place listed by one student and dark red for a place listed by many.
Each point has a dark outline, so the palest points still show on the light basemap.
Hover over a place for its name and its number of entries of each type; check locations before continuing.

In [ ]:
places.explore(
    column="popularity",
    cmap="Reds",
    marker_kwds={"radius": 6},
    style_kwds={"color": "#333333", "weight": 1, "fillOpacity": 0.9},   # a dark outline on every point
    tooltip=["name", "popularity", "eat", "study", "unwind"],
    tiles="Esri.WorldGrayCanvas",
)

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 3. Convert to feet

Buffers use the units of the CRS, so in `EPSG:4326`, `.buffer(1320)` would mean 1,320 **degrees**.
To measure distances, convert the layer to a **projected** CRS, whose units are a length.

Berkeley is in Zone 3 of the **California State Plane** system, which divides the state into six zones.
Its code in feet is [`EPSG:2227`](https://epsg.io/2227), NAD83 / California zone 3, measured in US survey feet.
Many local governments in this part of California publish their GIS data in it, so we use it here.
`.to_crs(2227)` converts the geometry coordinates and returns a new GeoDataFrame.
Save it under a new name ending in `_reproj_2227`, such as `places_reproj_2227`, so the name tells you which CRS a layer is in.

If you have no information about the local CRS for your data, `gdf.estimate_utm_crs()` gives you one in metres.
It picks the **UTM** zone your data falls in, one of 60 zones around the world; for Berkeley that is zone 10N, [`EPSG:32610`](https://epsg.io/32610) ([how UTM works](https://en.wikipedia.org/wiki/Universal_Transverse_Mercator_coordinate_system)).

In [ ]:
local_crs = 2227     # NAD83 / California zone 3, in US survey feet

places_reproj_2227 = places.to_crs(local_crs)
print(places_reproj_2227.crs.name)
print("unit:", places_reproj_2227.crs.axis_info[0].unit_name)
places_reproj_2227[["name", "geometry"]].head(3)

The geometry coordinates are now in feet, for example about 6,053,829 and 2,144,821 for Doe Library.
The original `lat` and `lon` columns remain in degrees; `.to_crs()` transforms the geometry column.
`places` itself is unchanged and still in degrees.
From here on, every layer we measure is reprojected first and named with `_reproj_2227`.

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 4. Three shapes that all mean "a quarter mile"

Here we use the UC Berkeley campus boundary, downloaded from OpenStreetMap with the `osmnx` package and saved as `data/raw/campus.parquet`.
You will learn to work with `osmnx` in the coming weeks.

Buffer the campus boundary in three ways, each by **1,320 ft**, a quarter mile:

- **Polygon:** grow the campus outward by 1,320 ft.
- **Boundary:** make a band 1,320 ft on each side of the campus edge.
- **Centroid:** make a 1,320 ft circle around the polygon's geometric centre.

Why a quarter mile?
A mile is 5,280 ft, so a quarter mile is 5,280 ÷ 4 = 1,320 ft and a half mile is 2,640 ft.
Planners treat a quarter mile (1,320 ft, about 400 m) as a 5-minute walk, the usual distance people will walk to a bus stop or a neighbourhood shop.
A half mile (2,640 ft, about 800 m) is a 10-minute walk and the usual catchment around a rail station.
This notebook uses both.

Name each buffer with its size and unit, such as `campus_buffer_1320_ft`, so you can tell the layers apart.
`.buffer(1320)` is a method, so it takes parentheses.
`.boundary`, `.centroid` and `.area` are properties, so they do not.

In [ ]:
campus = gpd.read_parquet("data/raw/campus.parquet")
campus_reproj_2227 = campus.to_crs(local_crs)

campus_buffer_1320_ft = campus_reproj_2227.buffer(1320)                    # the campus, grown outward by a quarter mile
campus_edge_buffer_1320_ft = campus_reproj_2227.boundary.buffer(1320)      # a band along the campus edge
campus_center_buffer_1320_ft = campus_reproj_2227.centroid.buffer(1320)    # a circle around the campus centroid

shapes = {
    "the campus": campus_reproj_2227.geometry.iloc[0],
    "the campus grown by 1,320 ft": campus_buffer_1320_ft.iloc[0],
    "a band along the edge": campus_edge_buffer_1320_ft.iloc[0],
    "a circle at the centroid": campus_center_buffer_1320_ft.iloc[0],
}

# .area is in square feet, and 43,560 square feet make one acre
for label, shape in shapes.items():
    print(label, ":", round(shape.area / 43_560), "acres,", places_reproj_2227.within(shape).sum(), "places inside")

The grown campus contains **135 of 157 places**, and the edge band contains 134.
The difference is the Big C, which is more than a quarter mile from the campus edge, so it falls in the hole in the middle of the band (see the middle map below).
The centroid circle contains **none**.

This campus polygon covers **1,284 acres**, including the eastern hill lands.
Its centroid lies about a mile east of Doe Library, far from most submitted places.
Check a boundary's extent before interpreting its centre.

The maps show the three shapes at the same scale.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 6))

campus_buffer_1320_ft.plot(
    ax=axes[0],
    color="#cfe3f7",
)
campus_edge_buffer_1320_ft.plot(
    ax=axes[1],
    color="#f2c14e",
)
campus_center_buffer_1320_ft.plot(
    ax=axes[2],
    color="#e4572e",
)

titles = [
    "the campus, grown outward",
    "a band along the edge",
    "a circle at the centroid",
]

# zoom to the grown campus, with 1,000 ft of margin; places farther away fall outside the frame
minx, miny, maxx, maxy = campus_buffer_1320_ft.total_bounds

# draw the campus outline and the places on every panel, then add the title
for i in range(3):
    campus_reproj_2227.plot(
        ax=axes[i],
        facecolor="none",
        edgecolor="#333333",
        linewidth=0.8,
    )
    places_reproj_2227.plot(
        ax=axes[i],
        color="#003262",
        markersize=6,
    )
    axes[i].set_xlim(minx - 1000, maxx + 1000)
    axes[i].set_ylim(miny - 1000, maxy + 1000)
    axes[i].set_title(titles[i])
    axes[i].set_axis_off()

fig.savefig(
    output_dir_vis / "campus_buffer_1320_ft.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

The middle panel has a hole, because the boundary buffer reaches only partway into the campus interior.
The polygon buffer includes the original campus area and extends outward.

<hr style="border: 1px solid #fdb515;" />

### ✏️ Your Turn: a wider net

Repeat this section at a **half mile, 2,640 ft**, by filling in each `...` in the cell below.

1. Buffer the campus three ways by 2,640 ft, as in the first cell of this section.
2. For each shape, compute its area in acres and the number of class places inside it.
3. For each shape, divide its half-mile area by its quarter-mile area, to see how many times larger it got.

Doubling the radius makes a circle four times larger, because a circle's area grows with the square of its radius.
Did the grown campus and the band also grow four times?
Did the number of places inside grow with the area?
Explain what you see in two or three sentences.

In [ ]:
# 1. buffer the campus three ways by a half mile, as in the first cell of this section
campus_buffer_2640_ft = ...          # the campus, grown outward
campus_edge_buffer_2640_ft = ...     # a band along the campus edge
campus_center_buffer_2640_ft = ...   # a circle around the campus centroid

half_mile = {
    "the campus grown": campus_buffer_2640_ft.iloc[0],
    "a band along the edge": campus_edge_buffer_2640_ft.iloc[0],
    "a circle at the centroid": campus_center_buffer_2640_ft.iloc[0],
}
quarter_mile = {
    "the campus grown": campus_buffer_1320_ft.iloc[0],
    "a band along the edge": campus_edge_buffer_1320_ft.iloc[0],
    "a circle at the centroid": campus_center_buffer_1320_ft.iloc[0],
}

# 2. the area in acres and the number of class places inside each half-mile shape
for label, shape in half_mile.items():
    acres = ...           # .area is in square feet, and 43,560 square feet make one acre
    places_inside = ...   # count the places .within() the shape
    print(label, ":", round(acres), "acres,", places_inside, "places inside")

# 3. how many times larger each half-mile shape is than its quarter-mile version
for label in half_mile:
    times_larger = ...    # the half-mile area divided by the quarter-mile area
    print(label, ":", round(times_larger, 1), "times larger")

*Your answer:*

<details>
<summary>Check your answer</summary>

| Shape | Quarter mile | Half mile | Times larger | Places inside at a half mile |
|---|---|---|---|---|
| the campus grown | 3,012 acres | 4,627 acres | 1.5 | 142 |
| a band along the edge | 2,796 acres | 4,627 acres | 1.7 | 142 |
| a circle at the centroid | 125 acres | 502 acres | 4.0 | 5 |

Only the circle grows four times.
The grown campus already contains the campus's own 1,284 acres, which do not grow, and the ring added around a large shape grows roughly with the distance, not its square.
At 2,640 ft the band is exactly as large as the grown campus, because a band that wide on each side of the edge covers the whole inside of the campus and the hole disappears.

The number of places barely follows the area.
135 of the 157 places are already within a quarter mile of the campus, so the half mile adds only 7.
The circle quadruples but still reaches only 5 places, because the centroid sits in the eastern hills, about a mile from Doe Library.

</details>

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 5. On or off campus?

For these points, [`.within(shape)`](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoSeries.within.html) returns `True` strictly inside the campus polygon.
Points exactly on its boundary return `False`.

Count the places of each type, and how many of them are on campus.
A place listed as two types counts once for each.

In [ ]:
# True for places inside the campus polygon
places_reproj_2227["on_campus"] = places_reproj_2227.within(campus_reproj_2227.geometry.iloc[0])

# True where at least one entry listed the place as that type
listed_as = places_reproj_2227[types] > 0

pd.DataFrame({
    "places": listed_as.sum(),
    "on_campus": listed_as[places_reproj_2227["on_campus"]].sum(),
})

Most study places are on campus; most eating places are outside it.
`.sum()` counts `True` values as 1.

Next, build clusters from the places themselves.

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 6. Hotspots for eating, studying and unwinding

Here a **cluster** is a connected area containing at least **3 places listed as the same type**.
Informally, each cluster is a hotspot, an area where the class goes to eat, study or unwind.
The word is used loosely here, since we count places without the statistical tests of a GIS hotspot analysis.

To build the clusters:

1. Buffer each place by **330 ft** (about 100 m), once for each type it was listed as, and keep that type as `cluster_type`.
2. `.dissolve(by="cluster_type")` merges overlapping buffers of each type.
3. `.explode()` separates disconnected pieces into rows.
4. Count same-type places inside each piece; keep pieces with at least 3.

Buffers overlap when centres are less than 660 ft apart.
Chains of overlaps can connect places much farther apart, so a cluster does not mean every pair is within 660 ft.

In [ ]:
# one buffer for each place and each type it was listed as
buffer_tables = []
for place_type in types:
    of_type = places_reproj_2227[places_reproj_2227[place_type] > 0]
    of_type_buffer_330_ft = gpd.GeoDataFrame(
        geometry=of_type.buffer(330),
        crs=local_crs,
    )
    of_type_buffer_330_ft["cluster_type"] = place_type
    buffer_tables.append(of_type_buffer_330_ft)
cluster_buffer_330_ft = pd.concat(buffer_tables)

clusters = cluster_buffer_330_ft.dissolve(by="cluster_type").reset_index()
clusters = clusters.explode(index_parts=False).reset_index(drop=True)

# count the places of the area's own type inside each area
places_of_type = []
for i in range(len(clusters)):
    same_type = places_reproj_2227[places_reproj_2227[clusters["cluster_type"].iloc[i]] > 0]
    places_of_type.append(same_type.within(clusters.geometry.iloc[i]).sum())
clusters["places_of_type"] = places_of_type

print(len(clusters), "areas before the rule")

# a cluster needs at least 3 places of its type
clusters = clusters.loc[clusters["places_of_type"] >= 3].copy().reset_index(drop=True)

clusters["cluster_id"] = clusters.index                       # a number for each cluster
clusters["area_acres"] = (clusters.area / 43_560).round(1)   # 43,560 square feet make one acre

print(len(clusters), "clusters")
clusters[["cluster_id", "cluster_type", "places_of_type", "area_acres"]]

The **52** pieces reduce to **7 clusters** (4 eat, 1 study and 2 unwind).

Plot them over cached OpenStreetMap roads and green spaces.
These panels focus on campus; the final interactive map also shows a small eating cluster on University Avenue, west of downtown.

In [ ]:
roads = gpd.read_parquet("data/raw/roads.parquet")
roads_reproj_2227 = roads.to_crs(local_crs)

green_spaces = gpd.read_parquet("data/raw/green_spaces.parquet")
green_spaces_reproj_2227 = green_spaces.to_crs(local_crs)

cluster_colors = {
    "eat": "#e4572e",
    "study": "#2e86ab",
    "unwind": "#5b8e3e",
}

# zoom every panel to the area around campus, centered on Doe Library
center = places_reproj_2227[places_reproj_2227["name"] == "Doe Library"].geometry.iloc[0]

fig, axes = plt.subplots(1, 3, figsize=(16, 6))

for i in range(3):
    cluster_type = types[i]
    color = cluster_colors[cluster_type]
    places_of_type = places_reproj_2227[places_reproj_2227[cluster_type] > 0]

    roads_reproj_2227.plot(
        ax=axes[i],
        color="#dddddd",
        linewidth=0.5,
    )
    green_spaces_reproj_2227.plot(
        ax=axes[i],
        color="#e3ede0",
    )
    campus_reproj_2227.boundary.plot(
        ax=axes[i],
        color="#888888",
        linewidth=1,
    )
    clusters[clusters["cluster_type"] == cluster_type].plot(
        ax=axes[i],
        color=color,
        alpha=0.4,
    )
    places_of_type.plot(
        ax=axes[i],
        color=color,
        markersize=places_of_type[cluster_type] * 12,   # bigger point = more entries of this type
    )

    axes[i].set_xlim(center.x - 5600, center.x + 4900)
    axes[i].set_ylim(center.y - 4300, center.y + 3600)
    axes[i].set_title(cluster_type + " clusters", fontsize=13)
    axes[i].set_axis_off()

fig.savefig(
    output_dir_vis / "clusters_by_type.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

The eating clusters are south of campus and downtown.
The study cluster covers central campus and reaches the Greek Theatre in the northeast, which one student listed for studying.
The unwind clusters are small, because parks and viewpoints are more scattered.

<hr style="border: 1px solid #fdb515;" />

### ✏️ Your Turn: a different distance

Set `test_distance` to **800**, in feet, and run the cell.
Then, in the map cell below it, fill in `test_clusters_of_type` by filtering `test_clusters` the same way the line above it filters `clusters`, and run it.
The map is saved as `clusters_330_vs_800_ft.png` in `vis/` and goes in your submission.
How many clusters remain, and how large are they?
What happens to the eating clusters south of campus and downtown?

In [ ]:
test_distance = ...

test_tables = []
for place_type in types:
    of_type = places_reproj_2227[places_reproj_2227[place_type] > 0]
    of_type_buffer_800_ft = gpd.GeoDataFrame(
        geometry=of_type.buffer(test_distance),
        crs=local_crs,
    )
    of_type_buffer_800_ft["cluster_type"] = place_type
    test_tables.append(of_type_buffer_800_ft)

test_clusters = pd.concat(test_tables).dissolve(by="cluster_type").reset_index()
test_clusters = test_clusters.explode(index_parts=False).reset_index(drop=True)

test_counts = []
for i in range(len(test_clusters)):
    same_type = places_reproj_2227[places_reproj_2227[test_clusters["cluster_type"].iloc[i]] > 0]
    test_counts.append(same_type.within(test_clusters.geometry.iloc[i]).sum())
test_clusters["places_of_type"] = test_counts

test_clusters = test_clusters[test_clusters["places_of_type"] >= 3].copy()
test_clusters["area_acres"] = (test_clusters.area / 43_560).round(1)
test_clusters[["cluster_type", "places_of_type", "area_acres"]]

In [ ]:
from matplotlib.patches import Patch

In [ ]:
# one panel per type, with your 800 ft clusters drawn lighter under the 330 ft clusters from section 6
fig, axes = plt.subplots(1, 3, figsize=(16, 6))

for i in range(3):
    cluster_type = types[i]
    color = cluster_colors[cluster_type]

    # the 330 ft clusters of this panel's type, from section 6
    clusters_of_type = clusters[clusters["cluster_type"] == cluster_type]
    # your 800 ft clusters of this panel's type, filtered from test_clusters in the same way
    test_clusters_of_type = ...

    roads_reproj_2227.plot(
        ax=axes[i],
        color="#dddddd",
        linewidth=0.5,
    )
    green_spaces_reproj_2227.plot(
        ax=axes[i],
        color="#e3ede0",
    )
    campus_reproj_2227.boundary.plot(
        ax=axes[i],
        color="#888888",
        linewidth=1,
    )
    test_clusters_of_type.plot(
        ax=axes[i],
        color=color,
        alpha=0.25,
    )
    clusters_of_type.plot(
        ax=axes[i],
        color=color,
        alpha=0.6,
    )
    places_reproj_2227[places_reproj_2227[cluster_type] > 0].plot(
        ax=axes[i],
        color="#333333",
        markersize=4,
    )

    # one legend per panel, in that panel's color
    axes[i].legend(
        handles=[
            Patch(color=color, alpha=0.6, label="330 ft buffers"),
            Patch(color=color, alpha=0.25, label=str(test_distance) + " ft buffers"),
        ],
        loc="lower left",
    )
    axes[i].set_xlim(center.x - 5600, center.x + 4900)
    axes[i].set_ylim(center.y - 4300, center.y + 3600)
    axes[i].set_title(cluster_type + " clusters", fontsize=13)
    axes[i].set_axis_off()

fig.savefig(
    output_dir_vis / "clusters_330_vs_800_ft.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

*Your answer:*

<details>
<summary>Check your answer</summary>

With 800 ft buffers, only **5 clusters** remain.
The eating clusters near campus and downtown merge into one area of about **672 acres** with 80 eating places.
The main study cluster grows to about 328 acres, a small study cluster appears downtown, and the unwind cluster grows to 548 acres.
Larger buffers join separate hotspots into one, so the distance you choose changes the result.

</details>

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 7. Which clusters does each place fall in?

A place may sit inside clusters of several types.
`gpd.sjoin(left, right, predicate="within")` returns a row for each left-hand point inside a right-hand cluster, carrying attributes from both.

Use `how="inner"` to keep matches only.
Both layers must use the same CRS.

In [ ]:
in_clusters = gpd.sjoin(
    places_reproj_2227,
    clusters,
    how="inner",
    predicate="within",
)

print("places:", len(places_reproj_2227))
print("rows after the join:", len(in_clusters))
matched_places = in_clusters["name"].nunique()
unmatched_places = len(places_reproj_2227) - matched_places
print("distinct matched places:", matched_places)
print("places with no cluster match:", unmatched_places)
in_clusters[["name", "eat", "study", "unwind", "cluster_type", "cluster_id"]].head()

**157 places produce 157 matches, and the equal numbers are a coincidence.**
Of the 157 places, 36 are in no cluster and another 36 appear in two overlapping clusters, so the missing rows and the extra rows cancel out.
After a spatial join, `len()` counts **matches**.
Counting distinct matched places with `in_clusters["name"].nunique()` gives **157 − 121 = 36 unmatched places**.
Subtracting the 157 match rows would give 0, the wrong answer, because some places match more than one cluster.

`in_clusters[types] > 0` marks the types each matched place was listed as.
Grouping those `True`/`False` columns by `cluster_id` and summing counts the places of each type in each cluster.
A place listed as two types counts in both columns, so the total number of places in each cluster comes from `.size()` instead.
`.join()` attaches those counts to the cluster summary using `cluster_id`.

In [ ]:
# True where the matched place was listed as that type
type_flags = in_clusters[types] > 0

cluster_mix = type_flags.groupby(in_clusters["cluster_id"]).sum()
cluster_mix["places"] = in_clusters.groupby("cluster_id").size()

cluster_table = clusters.set_index("cluster_id")[["cluster_type", "area_acres"]].join(cluster_mix)
cluster_table.sort_values("places", ascending=False).head()

The eating cluster south of campus has the most places, and it also contains 5 study and 9 unwind places.
The central study cluster is mainly study places.
Overlapping clusters show where uses mix.

These hotspots describe our class sample only; they are not zoning districts.
Straight-line buffers do not measure walking access; Week 13 uses the street network to build walksheds.

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 8. Final map

Put the clusters and the places on one interactive map.
`.explore()` returns a map, and passing that map as `m=` to a second `.explore()` draws the second layer on it.
`name=` labels each layer, and `folium.LayerControl()` adds a box in the top-right corner for turning layers on and off.
The basemap is added separately with `folium.TileLayer(..., control=False)`, so the box lists only the data layers.
A place listed as two types appears in both of their layers.
[Folium](https://python-visualization.github.io/folium/latest/) is the library `.explore()` uses to draw maps.

**Why this notebook uses Esri basemaps.**
Some basemap services now fail to work on offline maps, the HTML files you save and then open on your own computer or from Gradescope.
You saw this in Week 6, when the tiles in your saved maps did not load.
(Even a working basemap needs an internet connection, because the tiles are downloaded each time the map opens.)

- **CARTO:** since late August 2026, it requires an API key.
  Without one, its tiles come back with an "API KEY REQUIRED" watermark.
- **OpenStreetMap:** since around March 2026, every tile request must carry a Referer header, which names the web page asking for the tile.
  A file opened from your computer has no web address, so it sends none, and OpenStreetMap blocks it and shows an "Access Blocked" tile instead of the map.
- **Stadia,** the basemap you used in Week 6, refuses requests from saved files and shows "401 Error, Invalid Authentication" tiles.

Esri's gray tiles still load in saved files, so this notebook uses them for every map, including the maps you save and submit.

Hover over a cluster or a place for its details.

In [ ]:
import folium

In [ ]:
type_colors = [cluster_colors[t] for t in types]   # the section 6 colors, in the order eat, study, unwind

hotspot_map = clusters.explore(
    column="cluster_type",
    categories=types,
    cmap=type_colors,
    tooltip=["cluster_type", "places_of_type", "area_acres"],
    style_kwds={"fillOpacity": 0.3},
    legend_kwds={"caption": "place type"},
    name="clusters",
    tiles=None,
)

# one layer of places for each type
for place_type in types:
    places_reproj_2227[places_reproj_2227[place_type] > 0].explore(
        m=hotspot_map,
        color=cluster_colors[place_type],
        tooltip=["name", "popularity", "eat", "study", "unwind"],
        marker_kwds={"radius": 5},
        name=place_type + " places",
    )

campus_reproj_2227.boundary.explore(
    m=hotspot_map,
    color="#555555",
    name="campus boundary",
)

# the basemap, kept out of the layer box
folium.TileLayer(
    "Esri.WorldGrayCanvas",
    control=False,
).add_to(hotspot_map)

folium.LayerControl().add_to(hotspot_map)

hotspot_map.save(output_dir_vis / "campus_hotspots.html")
hotspot_map

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## ✅ Submit Part A in class (0.5 points)

In class, before **2:10 pm**:

1. Run your cells so the tables and maps show below them, including the map in the section 6 Your Turn.
2. **Save the notebook** (`Cmd + S` or `Ctrl + S`).
   The zip copies the file as it was last saved, and the check reads your answers from that saved copy.
3. Run the two cells below.
   They check that every Your Turn in Part A is done, then pack this notebook and the four Part A maps from `vis/` into `w7_s1_part_A_submission.zip`.
   If something is missing, the cell stops with a message that says what to finish.
4. In the file browser, right-click `w7_s1_part_A_submission.zip` and choose **Download**.
5. Upload the zip to the [Gradescope assignment](https://www.gradescope.com/courses/1333040/assignments/8805378).

That is the class activity done.
Part B is the optional bonus, for home.

In [ ]:
import json
import os
import zipfile
from datetime import datetime

In [ ]:
# find this notebook, even if it was renamed, e.g. "my_places_w7_s1 (1).ipynb"; the most recently saved copy wins
notebooks = sorted(Path(".").glob("*7_s1*.ipynb"), key=os.path.getmtime)
assert notebooks, "Can't find this notebook. Open it from its own folder and run this again."
notebook = notebooks[-1]

# every variable this notebook has created so far
defined = globals()

# stop with a message if a Your Turn is not done
assert "type_ratings" in defined and list(type_ratings.index.names) == ["name", "place_type"], (
    "Finish the section 2 Your Turn. Fill in the groupby with name, then place_type, and run the cell."
)
# the three half-mile buffers from the section 4 Your Turn, with their sizes in acres
half_mile_acres = {
    "campus_buffer_2640_ft": 4627,
    "campus_edge_buffer_2640_ft": 4627,
    "campus_center_buffer_2640_ft": 502,
}
for name, acres in half_mile_acres.items():
    buffer = defined.get(name)
    assert isinstance(buffer, gpd.GeoSeries) and round(buffer.area.sum() / 43_560) == acres, (
        "Finish the section 4 Your Turn. Make " + name + " a 2,640 ft buffer and run the cell."
    )
assert defined.get("test_distance") == 800, (
    "Finish the section 6 Your Turn. Set test_distance to 800 and run the cell."
)
assert (output_dir_vis / "clusters_330_vs_800_ft.png").exists(), (
    "Finish the section 6 Your Turn map. Fill in test_clusters_of_type and run the cell, including the savefig line."
)

# count the Part A answers that are still empty in the saved notebook
saved_cells = json.loads(notebook.read_text())["cells"]
empty_answers = 0
for cell in saved_cells:
    text = "".join(cell["source"])
    if cell["cell_type"] == "markdown" and "# Part B" in text:
        break
    if text.strip() == "*Your answer:*":
        empty_answers = empty_answers + 1
assert empty_answers == 0, (
    str(empty_answers) + " Part A answer(s) still empty. Write each one in its *Your answer:* cell, save the notebook and run this again."
)

saved = datetime.fromtimestamp(os.path.getmtime(notebook))
print("Notebook last saved at", saved.strftime("%H:%M:%S"), "- if that is not just now, save and run this again.")

# the four Part A maps and the cells that save them
maps = {
    "campus_buffer_1320_ft.png": "section 4",
    "clusters_by_type.png": "section 6",
    "clusters_330_vs_800_ft.png": "the section 6 Your Turn map",
    "campus_hotspots.html": "section 8",
}
for map_name, part in maps.items():
    assert (output_dir_vis / map_name).exists(), "Run " + part + " again, including the line that saves the map."

submission = Path("w7_s1_part_A_submission.zip")
with zipfile.ZipFile(submission, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(notebook)
    for map_name in maps:
        z.write(output_dir_vis / map_name)

with zipfile.ZipFile(submission) as z:
    print("Created", submission, "containing", z.namelist())

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

# Part B: Your Places

Part B is optional, for **2 bonus points**, due **Wednesday, October 7, at 12:30 pm**.
Work on it at home, in this same notebook.
It uses the tables and clusters from Part A, so if you restart the notebook, run Part A again first.

Use **your Lab 2 file, extended by five places**.
Preserve the original.
Copy it into this week's `data/raw/` folder, then edit the copy and save it with a name ending in `_extended.csv`.

The cell below uses the student course-folder layout.
Update the source path if your file is elsewhere.

In [ ]:
import shutil

# Replace the filename with your Lab 2 file's name.
source_path = Path("../../../lab/w02/nameofyourfile.csv")
extended_path = Path("data/raw") / (source_path.stem + "_extended.csv")
assert not extended_path.exists(), "This copy already exists; edit it instead of overwriting it."
shutil.copy2(source_path, extended_path)

Open the copied CSV with **Open With → Editor**.
Add five rows in the header's column order.
Quote values containing commas, such as `"2433 Durant Ave, Berkeley, CA 94704"`.
Google Maps shows coordinates as latitude, then longitude.
Save the edited copy.

<hr style="border: 1px solid #fdb515;" />

### ✏️ Question 1: prepare your places

Load your file and inspect it below.
Correct spelling variants and conflicting coordinates in your working copy before grouping.
Use the same address and name checks as section 2; different branches can share a name.

In [ ]:
my_path = ...   # the path to your _extended file

my_df = pd.read_csv(my_path)
my_df["name"] = my_df["name"].str.strip()
my_df["category"] = my_df["category"].str.strip().str.lower()
my_df.head()

Once the names are consistent, run the checks and build one row per place.
If coordinates disagree, resolve the location in your working copy and reload it.
`rows_in_file` preserves the number of entries for each place.

In [ ]:
# anything that is not a number becomes NaN, as in section 2
my_df["lat"] = pd.to_numeric(my_df["lat"], errors="coerce")
my_df["lon"] = pd.to_numeric(my_df["lon"], errors="coerce")
assert my_df["name"].notna().all() and my_df["name"].ne("").all(), "Fill in place names."
assert my_df["category"].isin(place_types).all(), "Use one of the seven categories in section 2."
assert my_df["lat"].between(-90, 90).all(), "Fix missing or invalid latitudes."
assert my_df["lon"].between(-180, 180).all(), "Fix missing or invalid longitudes."
my_coordinate_counts = my_df.groupby("name")[["lat", "lon"]].nunique(dropna=False)
assert my_coordinate_counts.eq(1).all().all(), "Resolve conflicting coordinates before grouping."

# one row per place, with one point per place, as in section 2
my_summary = my_df.groupby("name").agg(
    rows_in_file=("name", "size"),
    lat=("lat", "first"),
    lon=("lon", "first"),
)
# entries per place and type, as in section 2
my_df["place_type"] = my_df["category"].map(place_types)
my_type_counts = (
    my_df.groupby(["name", "place_type"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=types, fill_value=0)   # a type your file does not have becomes a column of 0s
)
my_summary = my_summary.join(my_type_counts).reset_index()

my_points = gpd.points_from_xy(
    x=my_summary["lon"],
    y=my_summary["lat"],
)

my_places = gpd.GeoDataFrame(
    my_summary,
    geometry=my_points,
    crs=4326,
)
my_places_reproj_2227 = my_places.to_crs(...)   # convert to the same CRS in feet as Part A

print(len(my_places_reproj_2227), "places")
print("places listed as each type:")
print((my_places_reproj_2227[types] > 0).sum())

my_places_reproj_2227.explore(
    color="#003262",
    tooltip=["name", "eat", "study", "unwind"],
    tiles="Esri.WorldGrayCanvas",
)

Check the map.
Are the points in the right locations?
How many places do you have, and how many of each type?

*Your answer:*

**Rate each use of your places separately.**
Do the same fix on your own places, and fill in the `groupby` as in the section 2 Your Turn.
In your own file you rated each place once, so each place gets one row with your rating, unless you listed the same place twice for two uses.
The fix matters when several people rate the same place, as in the class file, and this code is ready for that.

The first line of output counts entries whose rating is blank or is not one of the three labels, such as `8` or `7/10`.
Those entries are left out of the averages, as they were in the class file.

In [ ]:
# keep the entries rated love it, like it or just okay, as in section 2
my_df["rating"] = pd.Categorical(
    my_df["rating"].astype(str).str.strip().str.lower(),
    categories=rating_order,
    ordered=True,
)
print("entries without one of the three ratings:", my_df["rating"].isna().sum())
rated = my_df[my_df["rating"].notna()].copy()

rated["rating_num"] = rated["rating"].cat.codes + 1
rated["is_loved"] = (rated["rating"] == "love it").astype(int)

my_type_ratings = rated.groupby([..., ...]).agg(
    entries=("name", "size"),
    rating_mean=("rating_num", "mean"),
    share_loved=("is_loved", "mean"),
)
my_type_ratings

<hr style="border: 1px solid #fdb515;" />

### ✏️ Question 2: what is near the place you visit most?

Pick the place from your file that you **visit most**.
Draw a quarter-mile (1,320 ft) buffer and count your places inside it, using `.buffer()` and `.within()`.
Copy the place name exactly.

In [ ]:
most_visited_name = ...   # the exact name of the place you visit most

most_visited = my_places_reproj_2227.loc[my_places_reproj_2227["name"] == most_visited_name]
assert len(most_visited) == 1, "Choose one exact name from my_places_reproj_2227."
most_visited_buffer_1320_ft = most_visited.buffer(...).iloc[0]

my_inside_1320_ft = my_places_reproj_2227.within(...)
print(my_inside_1320_ft.sum(), "of my places are within a quarter mile, including the place itself")
my_places_reproj_2227[my_inside_1320_ft][["name", "eat", "study", "unwind"]]

How many of your **other** places are within a quarter mile of the place you visit most?
Are these places you walk between?

*Your answer:*

<hr style="border: 1px solid #fdb515;" />

### ✏️ Question 3: change the radius

Copy your Question 2 code into the cell below and change the radius to a half mile, 2,640 ft.
Name the new variables `most_visited_buffer_2640_ft` and `my_inside_2640_ft`, so the quarter-mile results stay available and the map below can draw both buffers.
How many of your places are inside now?
In section 4, doubling the radius made the circle four times larger.
Did your count of places grow four times too?
Why or why not?

In [ ]:
# Your code here

Run the cells below to map the place you visit most with both of its buffers.
The map uses your Question 2 and Question 3 variables and is saved as `my_most_visited_buffers_w7.png` in `vis/`.

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

In [ ]:
# the two buffers around the place you visit most
half_mile = gpd.GeoSeries([most_visited_buffer_2640_ft], crs=local_crs)
quarter_mile = gpd.GeoSeries([most_visited_buffer_1320_ft], crs=local_crs)

fig, ax = plt.subplots(figsize=(8, 8))

roads_reproj_2227.plot(
    ax=ax,
    color="#dddddd",
    linewidth=0.5,
)
green_spaces_reproj_2227.plot(
    ax=ax,
    color="#e3ede0",
)
campus_reproj_2227.boundary.plot(
    ax=ax,
    color="#888888",
    linewidth=1,
)
half_mile.plot(
    ax=ax,
    color="#fdb515",
    alpha=0.25,
)
quarter_mile.plot(
    ax=ax,
    color="#fdb515",
    alpha=0.55,
)
my_places_reproj_2227.plot(
    ax=ax,
    color="#003262",
    markersize=25,
)
most_visited.plot(
    ax=ax,
    color="#e4572e",
    marker="*",
    markersize=250,
)

# zoom to the half-mile buffer, with 600 ft of margin
minx, miny, maxx, maxy = half_mile.total_bounds
ax.set_xlim(minx - 600, maxx + 600)
ax.set_ylim(miny - 600, maxy + 600)
ax.set_title("Around " + most_visited_name, fontsize=13)
ax.set_axis_off()

ax.legend(
    handles=[
        Patch(color="#fdb515", alpha=0.55, label="a quarter mile (1,320 ft)"),
        Patch(color="#fdb515", alpha=0.25, label="a half mile (2,640 ft)"),
        Line2D([], [], marker="o", color="#003262", linestyle="", label="my places"),
        Line2D([], [], marker="*", color="#e4572e", markersize=14, linestyle="", label="the place I visit most"),
    ],
    loc="lower left",
)

fig.savefig(
    output_dir_vis / "my_most_visited_buffers_w7.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

*Your answer:*

<hr style="border: 1px solid #fdb515;" />

### ✏️ Question 4: how many of the class's places are near each of your places?

Draw a quarter-mile (1,320 ft) buffer around **every** one of your places, then join the class's `places_reproj_2227` table to your buffers, with a spatial join as in section 7.
Summarise the matches per buffer.

In [ ]:
my_places_buffer_1320_ft = gpd.GeoDataFrame(
    {"place_buffered": my_places_reproj_2227["name"]},
    geometry=my_places_reproj_2227.buffer(1320),
    crs=local_crs,
)

my_nearby_1320_ft = gpd.sjoin(
    places_reproj_2227,
    my_places_buffer_1320_ft,
    how="inner",
    predicate=...,
)
nearby_counts = my_nearby_1320_ft.groupby("place_buffered").agg(
    places=("name", "nunique"),
    listings=("popularity", "sum"),
)

# Keep buffers with no matches, giving them zero counts.
nearby_counts = nearby_counts.reindex(my_places_buffer_1320_ft["place_buffered"], fill_value=0)
nearby_counts

Map your buffers, colored by how many of the class's places fall inside each one.
The class's places are the small gray points, and hovering over a buffer shows its counts.
The map is saved as `my_nearby_class_places_w7.html` in `vis/`.

In [ ]:
# your buffers with their counts of class places
my_buffers_counted = my_places_buffer_1320_ft.join(
    nearby_counts,
    on="place_buffered",
)

nearby_map = my_buffers_counted.explore(
    column="places",
    cmap="Reds",
    tooltip=["place_buffered", "places", "listings"],
    style_kwds={"fillOpacity": 0.45, "color": "#333333", "weight": 1},
    legend_kwds={"caption": "class places within a quarter mile"},
    tiles="Esri.WorldGrayCanvas",
)

places_reproj_2227.explore(
    m=nearby_map,
    color="#555555",
    marker_kwds={"radius": 3},
    tooltip=["name", "popularity"],
)

nearby_map.save(output_dir_vis / "my_nearby_class_places_w7.html")
nearby_map

Which buffers contain the most and fewest class places?
Use the map to explain the difference.
A listed place can match itself.
`listings` counts student–place listings, so one student can contribute through several places.

`.reindex(..., fill_value=0)` keeps your places with no matches in the result.

*Your answer:*

<hr style="border: 1px solid #fdb515;" />

### ✏️ Question 5: which clusters are your places in?

Join your places to the class's clusters.
Use `how="inner"` when the question is about matching places and clusters; fill in the spatial predicate.

In [ ]:
my_in_clusters = gpd.sjoin(
    my_places_reproj_2227,
    clusters,
    how="inner",
    predicate=...,
)
my_unmatched_count = len(my_places_reproj_2227) - my_in_clusters["name"].nunique()
print("My places with no cluster match:", my_unmatched_count)
my_in_clusters[["name", "eat", "study", "unwind", "cluster_type", "cluster_id"]]

Which places are in eat, study or unwind clusters?
Does each cluster type match how you use the place?
The count of unmatched places is **total − distinct matched**.
To list their names, use `my_places_reproj_2227.loc[~my_places_reproj_2227["name"].isin(my_in_clusters["name"]), "name"]`.
A left join is useful when you want those unmatched places included in the joined table itself.

*Your answer:*

<hr style="border: 1px solid #fdb515;" />

### ✏️ Question 6: map and conclusion

Overlay your places on the clusters.
Fill in the column used to color the clusters, then save the interactive map.

In [ ]:
my_map = clusters.explore(
    column=...,
    categories=types,
    cmap=type_colors,   # the same colors as the section 8 map
    style_kwds={"fillOpacity": 0.35},
    tiles="Esri.WorldGrayCanvas",
)

my_places_reproj_2227.explore(
    m=my_map,
    color="#003262",
    tooltip=["name", "eat", "study", "unwind"],
    marker_kwds={"radius": 6},
)

my_map.save(output_dir_vis / "my_places_map_w7.html")
my_map

Do your places fall in the class's hotspots, or somewhere else?
Then name one thing a straight-line buffer does not account for when you actually walk there, such as hills, buildings or the street layout.
Answer in two or three sentences.

*Your answer:*

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## ✅ Package and submit the bonus (2 points)

Due **Wednesday, October 7, at 12:30 pm**:

1. Run your cells so the tables and maps show below them, including the maps in Questions 3, 4 and 6.
   If you restarted the notebook, run Part A first.
2. **Save the notebook** (`Cmd + S` or `Ctrl + S`).
   The zip copies the file as it was last saved, and the check reads your answers from that saved copy.
3. Run the two cells below.
   They check that every Part B question is done, then pack this notebook and your three Part B maps from `vis/` into `w7_s1_part_B_submission.zip`.
   If something is missing, the cell stops with a message that says what to finish.
4. In the file browser, right-click `w7_s1_part_B_submission.zip` and choose **Download**.
5. Upload the zip to the [bonus assignment on Gradescope](https://www.gradescope.com/courses/1333040/assignments/8805589).

In [ ]:
import json
import os
import zipfile
from datetime import datetime

In [ ]:
# find this notebook, even if it was renamed, e.g. "my_places_w7_s1 (1).ipynb"; the most recently saved copy wins
notebooks = sorted(Path(".").glob("*7_s1*.ipynb"), key=os.path.getmtime)
assert notebooks, "Can't find this notebook. Open it from its own folder and run this again."
notebook = notebooks[-1]

# every variable this notebook has created so far
defined = globals()

# stop with a message if a question is not done
checks = {
    "my_places_reproj_2227": "Question 1",
    "my_type_ratings": "the rating groupby after Question 1",
    "my_inside_1320_ft": "Question 2",
    "my_inside_2640_ft": "Question 3",
    "my_nearby_1320_ft": "Question 4",
    "my_in_clusters": "Question 5",
}
for name, part in checks.items():
    assert name in defined, "Finish " + part + ". Replace each ... in its cells and run them."

# your three Part B maps and the questions that make them
maps = {
    "my_most_visited_buffers_w7.png": "the Question 3 map",
    "my_nearby_class_places_w7.html": "the Question 4 map",
    "my_places_map_w7.html": "Question 6",
}
for map_name, part in maps.items():
    assert (output_dir_vis / map_name).exists(), "Finish " + part + ". Run its cell, including the line that saves the map."

# count the Part B answers that are still empty in the saved notebook
saved_cells = json.loads(notebook.read_text())["cells"]
in_part_b = False
empty_answers = 0
for cell in saved_cells:
    text = "".join(cell["source"])
    if cell["cell_type"] == "markdown" and "# Part B" in text:
        in_part_b = True
    if in_part_b and text.strip() == "*Your answer:*":
        empty_answers = empty_answers + 1
assert empty_answers == 0, (
    str(empty_answers) + " Part B answer(s) still empty. Write each one in its *Your answer:* cell, save the notebook and run this again."
)

saved = datetime.fromtimestamp(os.path.getmtime(notebook))
print("Notebook last saved at", saved.strftime("%H:%M:%S"), "- if that is not just now, save and run this again.")

submission = Path("w7_s1_part_B_submission.zip")
with zipfile.ZipFile(submission, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(notebook)
    for map_name in maps:
        z.write(output_dir_vis / map_name)

with zipfile.ZipFile(submission) as z:
    print("Created", submission, "containing", z.namelist())

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Quick reference

| Task | Pattern |
|---|---|
| Clean and compare names | `.str.strip()`, `.str.lower()`, `.replace(...)` |
| One row per place | `df.groupby("name").agg(...)` |
| Create geometry after cleaning | `gpd.points_from_xy(lon, lat)`; `gpd.GeoDataFrame(..., crs=4326)` |
| Measure in feet, in Berkeley | `gdf_reproj_2227 = gdf.to_crs(2227)` |
| Measure in metres, when you do not know the local CRS | `gdf.to_crs(gdf.estimate_utm_crs())` |
| Buffer different shapes | `.buffer(1320)`, `.boundary.buffer(1320)`, `.centroid.buffer(1320)` |
| Name a buffer by its size and unit | `gdf_buffer_1320_ft = gdf.buffer(1320)` |
| Test points inside a shape | `.within(shape)`; excludes points on the boundary |
| Build connected areas by type | `.dissolve(by=...)`, then `.explode()` |
| Match places to clusters | `gpd.sjoin(..., predicate="within", how="inner")` |
| Count each pair of values, one column per value | `.groupby([a, b]).size().unstack(fill_value=0)` |
| Draw layers on one map | `m = gdf.explore(name=...)`, then `other.explore(m=m, name=...)`; `folium.LayerControl().add_to(m)` |

**Remember:** a row after a spatial join is a match.
Count unique place names when the question is about distinct places.